In [11]:





import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

df = pd.read_csv("creditcard.csv")
assert df.shape[0] == 284807 and (df["Class"] == 1).sum() == 492
df = df.dropna().drop_duplicates()
X, y = df.drop("Class", axis=1), df["Class"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42)
print(X_train.shape, int(y_train.sum()), X_test.shape, int(y_test.sum()))  # expect 226980/378 and 56746/95

pre = ColumnTransformer([('scale', StandardScaler(), ['Time', 'Amount'])], remainder='passthrough')
pipe = ImbPipeline([('preprocessor', pre),
                    ('smote', SMOTE(random_state=42)),
                    ('classifier', RandomForestClassifier(random_state=42, n_jobs=-1))])
grid = {'classifier__n_estimators': [100],
        'classifier__max_depth': [10, 20],
        'classifier__min_samples_split': [2, 5]}
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

gs = GridSearchCV(pipe, grid, cv=cv, scoring='average_precision',
                  n_jobs=1, refit=False, verbose=2)   # n_jobs=1 avoids nested-parallel slowdown; refit=False skips the extra final fit
gs.fit(X_train, y_train)

out = pd.DataFrame(gs.cv_results_)[
    ['param_classifier__max_depth', 'param_classifier__min_samples_split',
     'mean_test_score', 'std_test_score', 'rank_test_score']]
print(out.round(4))
out.to_csv("rf_grid_cv_ap.csv", index=False)

(226980, 30) 378 (56746, 30) 95
Fitting 3 folds for each of 4 candidates, totalling 12 fits
[CV] END classifier__max_depth=10, classifier__min_samples_split=2, classifier__n_estimators=100; total time= 2.2min
[CV] END classifier__max_depth=10, classifier__min_samples_split=2, classifier__n_estimators=100; total time= 3.7min
[CV] END classifier__max_depth=10, classifier__min_samples_split=2, classifier__n_estimators=100; total time= 4.1min
[CV] END classifier__max_depth=10, classifier__min_samples_split=5, classifier__n_estimators=100; total time= 2.3min
[CV] END classifier__max_depth=10, classifier__min_samples_split=5, classifier__n_estimators=100; total time= 2.1min
[CV] END classifier__max_depth=10, classifier__min_samples_split=5, classifier__n_estimators=100; total time= 2.2min
[CV] END classifier__max_depth=20, classifier__min_samples_split=2, classifier__n_estimators=100; total time= 2.9min
[CV] END classifier__max_depth=20, classifier__min_samples_split=2, classifier__n_estimat

### Saving and Loading Data to Google Drive for Persistence

To avoid re-uploading your dataset every time you start a new Colab session, you can save it to your Google Drive. Then, in any future session, you can mount your Drive and load the dataset directly.

In [12]:
import sklearn, imblearn, xgboost, platform, subprocess
info = (f"scikit-learn {sklearn.__version__} | imbalanced-learn {imblearn.__version__} | "
        f"xgboost {xgboost.__version__} | numpy {np.__version__} | pandas {pd.__version__}\n"
        f"{platform.platform()}\n"
        + subprocess.run("nproc; free -h | head -2", shell=True, capture_output=True, text=True).stdout)
print(info); open("environment.txt", "w").write(info)

scikit-learn 1.6.1 | imbalanced-learn 0.14.2 | xgboost 3.4.1 | numpy 2.1.3 | pandas 2.2.3
Linux-6.6.122+-x86_64-with-glibc2.39
2
               total        used        free      shared  buff/cache   available
Mem:            12Gi       2.0Gi       8.3Gi       2.4Mi       2.6Gi        10Gi



291

In [17]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import LinearSVC
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    average_precision_score, roc_auc_score, precision_score,
    recall_score, f1_score, matthews_corrcoef
)

tune_specs = {
    "Decision Tree": (
        DecisionTreeClassifier(random_state=42),
        {
            'classifier__max_depth': [10, 20],
            'classifier__min_samples_split': [2, 5]
        }
    ),

    "LinearSVC": (
        LinearSVC(max_iter=5000, random_state=42),
        {
            'classifier__C': [0.01, 0.1, 1.0, 10.0]
        }
    ),

    "MLP": (
        MLPClassifier(
            hidden_layer_sizes=(64, 32),
            activation='relu',
            solver='adam',
            max_iter=200,
            early_stopping=True,
            n_iter_no_change=15,
            validation_fraction=0.1,
            random_state=42
        ),
        {
            'classifier__hidden_layer_sizes': [(64, 32), (128, 64)],
            'classifier__alpha': [1e-4, 1e-2]
        }
    ),

    "XGBoost": (
        XGBClassifier(
            n_estimators=100,
            random_state=42,
            n_jobs=-1
        ),
        {
            'classifier__max_depth': [4, 6],
            'classifier__learning_rate': [0.1, 0.2]
        }
    ),
}

tuned_rows = []

for name, (est, grid_t) in tune_specs.items():

    print(f"\nTuning {name} ...")

    pipe_t = ImbPipeline([
        ('preprocessor', pre),
        ('smote', SMOTE(random_state=42)),
        ('classifier', est)
    ])

    gs_t = GridSearchCV(
        pipe_t,
        grid_t,
        cv=cv,
        scoring='average_precision',
        n_jobs=1,
        refit=True
    )

    gs_t.fit(X_train, y_train)

    cvr = pd.DataFrame(gs_t.cv_results_)

    cvr.to_csv(
        f"cv_{name.replace(' ', '_')}.csv",
        index=False
    )

    print(
        cvr[
            ['params', 'mean_test_score',
             'std_test_score', 'rank_test_score']
        ].round(4).to_string()
    )

    # Score predictions
    if hasattr(gs_t, "predict_proba"):
        s = gs_t.predict_proba(X_test)[:, 1]
    else:
        s = gs_t.decision_function(X_test)

    yp = gs_t.predict(X_test)

    tuned_rows.append({
        "Model": name,
        "Best params": str(gs_t.best_params_),
        "CV AP": round(gs_t.best_score_, 4),
        "Dev AP": round(
            average_precision_score(y_test, s), 4
        ),
        "ROC-AUC": round(
            roc_auc_score(y_test, s), 4
        ),
        "Prec": round(
            precision_score(y_test, yp, zero_division=0), 4
        ),
        "Recall": round(
            recall_score(y_test, yp, zero_division=0), 4
        ),
        "F1": round(
            f1_score(y_test, yp, zero_division=0), 4
        ),
        "MCC": round(
            matthews_corrcoef(y_test, yp), 4
        )
    })

tuned_df = pd.DataFrame(tuned_rows)

print("\n===== TUNED MODEL RESULTS =====")
print(tuned_df.to_string(index=False))

tuned_df.to_csv(
    "tuned_other_models.csv",
    index=False
)


Tuning Decision Tree ...
                                                              params  mean_test_score  std_test_score  rank_test_score
0  {'classifier__max_depth': 10, 'classifier__min_samples_split': 2}           0.4546          0.0402                2
1  {'classifier__max_depth': 10, 'classifier__min_samples_split': 5}           0.4597          0.0493                1
2  {'classifier__max_depth': 20, 'classifier__min_samples_split': 2}           0.3426          0.0370                3
3  {'classifier__max_depth': 20, 'classifier__min_samples_split': 5}           0.3414          0.0363                4

Tuning LinearSVC ...
                    params  mean_test_score  std_test_score  rank_test_score
0  {'classifier__C': 0.01}           0.7328          0.0374                1
1   {'classifier__C': 0.1}           0.7325          0.0365                4
2   {'classifier__C': 1.0}           0.7326          0.0366                2
3  {'classifier__C': 10.0}           0.7326      

In [22]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import precision_recall_curve


INCLUDE_RF = True   # set True only if you have time; Random Forest is the slow one

def best_thr(y_true, scores):
    p, r, t = precision_recall_curve(y_true, scores)
    f = np.where((p[:-1] + r[:-1]) == 0, 0, 2 * p[:-1] * r[:-1] / (p[:-1] + r[:-1]))
    i = int(np.argmax(f)); return t[i], f[i]

base_models = {
    "Decision Tree": DecisionTreeClassifier(max_depth=10, min_samples_split=2, random_state=42),
    "LinearSVC": LinearSVC(C=1.0, max_iter=5000, random_state=42),
    "MLP": MLPClassifier(
    hidden_layer_sizes=(64, 32),
    activation='relu',
    solver='adam',
    max_iter=200,
    early_stopping=True,
    n_iter_no_change=15,
    validation_fraction=0.1,
    random_state=42
),
    "XGBoost": XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=42, n_jobs=-1),
}
if INCLUDE_RF:
    base_models["Random Forest"] = RandomForestClassifier(n_estimators=100, max_depth=20,
                                                          min_samples_split=2, random_state=42, n_jobs=-1)

thr_rows = []
for name, clf in base_models.items():
    print(f"{name} ...")
    pipe_o = ImbPipeline([('preprocessor', pre), ('smote', SMOTE(random_state=42)), ('classifier', clf)])
    use_proba = hasattr(clf, "predict_proba")
    oof = cross_val_predict(pipe_o, X_train, y_train, cv=cv, n_jobs=1,
                            method="predict_proba" if use_proba else "decision_function")
    if use_proba: oof = oof[:, 1]
    thr, f1_oof = best_thr(y_train, oof)

    pipe_o.fit(X_train, y_train)
    s_dev = pipe_o.predict_proba(X_test)[:, 1] if use_proba else pipe_o.decision_function(X_test)
    native = 0.5 if use_proba else 0.0
    row = {"Model": name, "OOF-selected cutoff": round(float(thr), 4), "OOF F1 (train)": round(float(f1_oof), 4)}
    for tag, c in [("Native", native), ("OOF cutoff", thr)]:
        yp = (s_dev >= c).astype(int)
        row[f"{tag} Prec"] = round(precision_score(y_test, yp, zero_division=0), 4)
        row[f"{tag} Rec"] = round(recall_score(y_test, yp), 4)
        row[f"{tag} F1"] = round(f1_score(y_test, yp), 4)
    thr_rows.append(row)

thr_df = pd.DataFrame(thr_rows)
print(thr_df.to_string()); thr_df.to_csv("threshold_from_oof.csv", index=False)


Decision Tree ...
LinearSVC ...
MLP ...
XGBoost ...
Random Forest ...
           Model  OOF-selected cutoff  OOF F1 (train)  Native Prec  Native Rec  Native F1  OOF cutoff Prec  OOF cutoff Rec  OOF cutoff F1
0  Decision Tree               0.9987          0.6514       0.0916      0.7895     0.1641           0.5546          0.6947         0.6168
1      LinearSVC               1.8925          0.7612       0.0689      0.8632     0.1276           0.7935          0.7684         0.7807
2            MLP               0.9987          0.8183       0.6818      0.7895     0.7317           0.9298          0.5579         0.6974
3        XGBoost               0.9711          0.8478       0.3805      0.8211     0.5200           0.9452          0.7263         0.8214
4  Random Forest               0.7400          0.8413       0.8488      0.7684     0.8066           0.9296          0.6947         0.7952
